# Final project ver2.0 — 비교 실험

> **2026-11-06 oneway·mutual A/B/C1 본실험:** `Mutual_oneway_C1_20261106.ipynb`를 사용하세요. 이 노트북은 범용 개발용이며 기본 모델·전략 설정이 본실험과 다릅니다. 자세한 순서는 저장소의 `research/3_colab_guide.md`에 있습니다.

    **Final project ver2.0** — Windows + VS Code + Colab GPU.

    로컬 파일과 Colab 서버는 서로 다른 컴퓨터입니다. VS Code에서 `Select Kernel → Colab`을 선택하고 GPU를 확인하세요. 처음에는 `RUN_CONDITIONS=["D"]`로 기반을 확인합니다. **A·B·D는 제공하는 완성 코드**입니다. A/B에는 저장 VLA 모델 경로만 지정하면 됩니다. C는 학생의 루프를 연결합니다. 기존 `Final_project.ipynb`와 연구 조건의 의미가 다릅니다.

    씬·시작 명령·내부 프롬프트·루프 입출력의 자세한 순서는 **`docs/FINAL_PROJECT_V2_STUDENT_GUIDE.md`**를 먼저 읽으세요. 내부 프롬프트는 `final_project_v2/prompts/starter_strategy.txt`, C 루프는 `final_project_v2/student_strategy.py`입니다. B와 C는 기본적으로 같은 TXT를 읽습니다.

    기본 생성 씬은 시작 예시입니다. 기하 검사 결과는 실제 파지 불가능의 정답이 아닙니다. 실제 실행 영상으로 확인하세요. 안내는 `docs/FINAL_PROJECT_V2.md`에 있습니다.

로컬 GPU: `VLA2026 (conda, Python 3.12, CUDA)` 커널을 선택합니다. `SMOKE_TEST=False`의 기본 예산은 300초/6000스텝입니다. 연결 확인만 할 때 `SMOKE_TEST=True`를 사용하세요.


In [ ]:
import os, sys, json, platform, subprocess, shutil, time, uuid
from pathlib import Path
IS_COLAB = platform.system() == 'Linux' and Path('/content').is_dir()
TEAM = 'team01'
SOURCE_ROOT = os.environ.get('FPV2_SOURCE', '')  # 업로드 후 실제 원격 vla_pipeline 경로
REPO_URL = 'https://github.com/semin126/VLA2026.git'
REPO_REF = os.environ.get('FPV2_REPO_REF', 'instructor/runtime-conda-fixes')                 # 검증된 강사 배포 브랜치/commit
INSTALL = IS_COLAB and os.environ.get('FPV2_SKIP_INSTALL') != '1'
USE_DRIVE = IS_COLAB and os.environ.get('FPV2_NO_DRIVE') != '1'
PYTHON_OVERRIDE = os.environ.get('FPV2_PYTHON', '')
WORK = Path(os.environ.get('FPV2_WORK', '/content/Final_project_v2' if IS_COLAB else 'outputs/final_project_v2'))
SCENES_ROOT = os.environ.get('FPV2_SCENES', '')   # 비워두면 아래 'validation suite 생성' 셀이 이 세션 경로로 자동 지정
CUSTOM_SCENE = os.environ.get('FPV2_SCENE', '')    # 직접 저장한 씬 JSON. 비우면 개발 open
TASK_COMMAND = ''                              # 시작 명령. 비우면 scene.json의 command 사용
RUN_CONDITIONS = os.environ.get('FPV2_CONDITIONS', 'D').split(',')                         # A,B,C,D. E는 로컬 전용 셀
SMOKE_TEST = os.environ.get('FPV2_SMOKE', '0') == '1'  # 짧은 연결 검사만 True
SECONDS = float(os.environ.get('FPV2_SECONDS', '12' if SMOKE_TEST else '300'))
MAX_STEPS = int(os.environ.get('FPV2_MAX_STEPS', '2400' if SMOKE_TEST else '6000'))
ACTION_STEPS = int(os.environ.get('FPV2_ACTION_STEPS', '240'))
OBSERVER_STEPS = int(os.environ.get('FPV2_OBSERVER_STEPS', '60'))
MAX_CALLS = int(os.environ.get('FPV2_MAX_CALLS', '8'))
REPEAT_SEEDS = '42'
VLA_SCENE = os.environ.get('FPV2_VLA_SCENE', '')                                 # A: 전체 과제 명령을 학습한 모델
VLA_SUBTASK = os.environ.get('FPV2_VLA_SUBTASK', '')                               # B/C: 부분 운반 지시를 학습한 같은 모델
MODEL_SOURCE = os.environ.get('FPV2_MODEL_SOURCE', 'auto')  # auto: models 우선, 없으면 Hub
MODELS_DIR = os.environ.get('FPV2_MODELS_DIR', 'models')
VLM_MODEL = os.environ.get('FPV2_VLM_MODEL', '')  # 비우면 models/vlm 선택; Hub 모드/ID 지정도 가능
STRATEGY_OVERRIDE = os.environ.get('FPV2_STRATEGY', '')
B_STRATEGY_OVERRIDE = os.environ.get('FPV2_B_STRATEGY', '')
FINAL_TEST = False                            # 전략·모델 확정 후에만 True

VLM_SOURCE = os.environ.get('FPV2_VLM_SOURCE', MODEL_SOURCE)
VLA_SOURCE = os.environ.get('FPV2_VLA_SOURCE', MODEL_SOURCE)


## 코드·실행 환경·저장 위치

아직 push하지 않은 코드는 `Upload to Colab` 후 `SOURCE_ROOT`로 지정합니다. 기존 작업 폴더를 자동 pull/reset하지 않습니다. Linux에서는 독립 Python 3.12 환경과 기존 잠금 파일을 사용합니다.

Drive는 각 명령 종료 시 해시를 확인하여 백업합니다. 세션이 도중에 사라지면 미복사 파일은 잃을 수 있습니다. 모델 로딩 시간은 과제 시간과 별도로 기록합니다.


In [ ]:
GIT = shutil.which('git') or (str(Path(os.environ.get('ProgramFiles', 'C:/Program Files'))/'Git/cmd/git.exe') if platform.system() == 'Windows' else 'git')
if SOURCE_ROOT:
    PROJECT = Path(SOURCE_ROOT).expanduser().resolve()
else:
    candidates = []
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidates += [base, base/'vla_pipeline', base/'2026/VLA2026/vla_pipeline']
    PROJECT = next((p for p in candidates if (p/'scripts/final_project_v2.py').is_file()), None)
    if PROJECT is None:
        checkout = Path('/content' if IS_COLAB else '.') / ('fpv2_checkout_'+uuid.uuid4().hex[:8])
        subprocess.run([GIT, 'clone', REPO_URL, str(checkout)], check=True)
        subprocess.run([GIT, '-C', str(checkout), 'checkout', REPO_REF], check=True)
        PROJECT = checkout/'vla_pipeline'
PROJECT = PROJECT.resolve()
if not (PROJECT/'vlapipe/final_project_v2/runner.py').is_file():
    raise RuntimeError('ver2.0 코드가 없습니다. 최신 브랜치 또는 업로드 위치를 확인하세요.')
sys.path.insert(0, str(PROJECT))
try:
    from vlapipe.final_project_v2.notebook_runtime import (
        NotebookRun, prepare_python, select_scene, source_version,
        check_research_budget, build_student_validation_suite)
except ImportError as exc:
    raise RuntimeError('현재 PROJECT의 코드가 오래되었습니다. c-loop-validation 수정본을 받고 SOURCE_ROOT/REPO_REF를 확인하세요.') from exc
if not WORK.is_absolute():
    WORK = PROJECT/WORK
if USE_DRIVE:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
    BACKUP = Path('/content/drive/MyDrive/VLA2026/Final_project_v2')
else:
    BACKUP = None
run = NotebookRun(PROJECT, WORK, BACKUP, TEAM)
PY = prepare_python(PROJECT, WORK, install=INSTALL, python_override=PYTHON_OVERRIDE)
run.python = PY
B_STRATEGY = Path(B_STRATEGY_OVERRIDE) if B_STRATEGY_OVERRIDE else PROJECT/'final_project_v2/baseline_strategy.py'
C_STRATEGY = Path(STRATEGY_OVERRIDE) if STRATEGY_OVERRIDE else PROJECT/'final_project_v2/student_strategy.py'
DATA_STRATEGY = B_STRATEGY  # 같은 TXT 전략으로 초기 VLM 학습 후보 생성
SOURCE_VERSION = source_version(PROJECT)
print('SOURCE VERSION:', json.dumps(SOURCE_VERSION, ensure_ascii=False))
(run.root/'source_version.json').write_text(json.dumps(SOURCE_VERSION, indent=2), encoding='utf-8')
print('PROJECT:', PROJECT, '\nPYTHON:', PY, '\nRUN:', run.root, '\nBACKUP:', run.backup)
run.run('scripts/check_env.py')

print('Kernel Python:', sys.executable, '\nExecution Python:', PY)

from vlapipe.final_project_v2.model_sources import resolve_model


## 배포 모델 받기

처음에는 아래 DOWNLOAD_MODELS를 True로 바꿔 실행합니다. 새 버전으로 교체할 때는 UPGRADE_MODELS도 True로 설정합니다. 새 파일 검증이 끝난 뒤 이전 모델을 models/_previous/에 백업합니다. 모델 폴더가 준비되면 기본 auto 모드에서 로컬 모델을 사용합니다. 버전별 실제 검증 범위는 models/release_manifest.json과 docs/MODEL_RELEASE.md를 읽으세요.


In [ ]:
DOWNLOAD_MODELS = os.environ.get('FPV2_DOWNLOAD_MODELS', '0') == '1'
UPGRADE_MODELS = os.environ.get('FPV2_UPGRADE_MODELS', '0') == '1'
if DOWNLOAD_MODELS:
    download_options = ['--models-dir', MODELS_DIR]
    if UPGRADE_MODELS:
        download_options.append('--upgrade')
    run.run('scripts/download_models.py', *download_options)


In [ ]:
# 학생의 세 씬을 새 frozen suite로 등록합니다. 기존 결과는 삭제하지 않습니다.
# SCENES_ROOT를 지정했다면 그 suite를 그대로 사용합니다.
if not SCENES_ROOT:
    VALID_SUITE = build_student_validation_suite(run)
    SCENES_ROOT = str(VALID_SUITE)
    print('Validation suite:', VALID_SUITE)
else:
    print('Using existing SCENES_ROOT:', SCENES_ROOT)


In [ ]:
if SCENES_ROOT:
    SCENES = Path(SCENES_ROOT).expanduser()
    SCENES = (SCENES if SCENES.is_absolute() else PROJECT/SCENES).resolve()
else:
    SCENES = run.root/'scenes'
    run.run('scripts/final_project_v2.py', 'scenes', '--out', SCENES, '--tests', 10)
registry = json.loads((SCENES/'suite.json').read_text(encoding='utf-8'))
DEV_SCENES = [SCENES/r['path'] for r in registry['scenes'] if r['split']=='development']
VALID_SCENES = [SCENES/r['path'] for r in registry['scenes'] if r['split']=='validation']
print('Scenes:', SCENES, '\nDevelopment:', *DEV_SCENES, '\nValidation:', *VALID_SCENES, sep='\n')
from IPython.display import display, Image, HTML
import base64
def show_video(path):
    encoded = base64.b64encode(Path(path).read_bytes()).decode('ascii')
    display(HTML('<video controls width="700"><source src="data:video/mp4;base64,'+encoded+'" type="video/mp4"></video>'))
# development 씬이 있으면 미리보기, 없으면(=validation 전용 suite) validation 씬을 미리보기.
_preview = DEV_SCENES or VALID_SCENES
if _preview:
    display(Image(filename=str(_preview[0].with_suffix('')/'scene.png'), width=700))
else:
    print('미리볼 씬이 없습니다. suite.json 의 split 을 확인하세요.')


In [ ]:
# 진단용 가짜 모델: C 연결 검사이며 정확도 실험이 아닙니다.
if VALID_SCENES:
    run.run('scripts/check_student_strategy_loop.py', '--scenes', SCENES,
            '--out', run.root/('check_c_loop_'+uuid.uuid4().hex[:8]))
else:
    print('validation 씬이 없어 C 진단을 건너뜁니다. 별도 validation suite를 지정하세요.')


## A~E 비교

|조건|처리|
|---|---|
|A|전체 과제 명령을 VLA에 직접 전달. 물체 선택의 정답 보조 없음|
|B|초기 이미지로 VLM 1회 계획, VLA가 순차 실행|
|C|B와 같은 초기 프롬프트·모델로 시작하고 학생 루프로 재관측·재계획|
|D|정답 위치 기반의 결정적 규칙＋고전 제어기(D-state, CV 아님)|
|E|Windows 로컬에서 사람의 조이스틱 조작|

같은 고정 초기 상태와 과제를 사용합니다. 물체 ACC·구역 macro-F1은 제한 시간 종료 시 물리 상태로 계산합니다. 추론과 제어는 과제 시간에 포함합니다. 기한을 넘은 추론은 초과를 기록하고 늦은 동작은 실행하지 않습니다. 제어는 20Hz 상한입니다.

A의 전체 명령과 B/C의 부분 명령에 대한 학습 경험이 필요합니다. 학습되지 않은 A의 실패를 계층 구조의 효과로 단정하지 않습니다. 개발 씬 3개에서 확인한 뒤 고정한 혼합 시험 씬 10개 이상을 평가하세요.


In [ ]:
check_research_budget(RUN_CONDITIONS, SECONDS, MAX_STEPS, ACTION_STEPS, MAX_CALLS, smoke=SMOKE_TEST)
SCENE = select_scene(PROJECT, SCENES, registry, CUSTOM_SCENE)
TRIALS = run.root/('trials_'+uuid.uuid4().hex[:8])  # 이 셀 재실행 시 새 비교 폴더
def trial(condition, scene=None):
    scene = Path(scene) if scene is not None else SCENE
    checkpoint = VLA_SCENE if condition=='A' else VLA_SUBTASK
    args = ['trial', '--condition', condition, '--scene', scene,
            '--out', TRIALS/Path(scene).stem/condition, '--seconds', SECONDS,
            '--max-steps', MAX_STEPS, '--action-steps', ACTION_STEPS,
            '--observer-steps', OBSERVER_STEPS, '--max-calls', MAX_CALLS,
            '--model-source', MODEL_SOURCE, '--models-dir', MODELS_DIR,
            '--vlm-source', VLM_SOURCE, '--vla-source', VLA_SOURCE]
    if condition in 'ABC' and checkpoint:
        args += ['--checkpoint', checkpoint]
    if condition in 'BC':
        selected_strategy = B_STRATEGY if condition=='B' else C_STRATEGY
        args += ['--strategy', selected_strategy]
        if VLM_MODEL:
            args += ['--vlm-model', VLM_MODEL]
    if TASK_COMMAND.strip():
        args += ['--task-command', TASK_COMMAND]
    if FINAL_TEST:
        args += ['--final-test']
    run.run('scripts/final_project_v2.py', *args)


**여기서 바꿀 것:** `SCENE=DEV_SCENES[0/1/2]`로 open/blocked/mixed를 선택하거나, 설정 셀의 `CUSTOM_SCENE`에 저장한 JSON 경로를 넣습니다. 시작 명령은 `TASK_COMMAND` 또는 JSON의 `command`입니다. `TASK_COMMAND`가 있으면 그것을 A~E에 공통 적용합니다. 현재 채점 과제는 모든 숫자 물체의 고정 구역 분류이며, 문장만 바꾸어 목표 구역이나 채점 대상을 변경하지 않습니다.

**A·B·D 실행:** `RUN_CONDITIONS=['A','B','D']`, `VLA_SCENE`, `VLA_SUBTASK`를 설정하세요. B는 완성된 `baseline_strategy.py`를 사용하므로 C의 빈칸이 있어도 실행됩니다. **C 연결 확인만 먼저 할 때:** `C_STRATEGY=PROJECT/'final_project_v2/loop_example.py'`로 명시 선택합니다. 학생 연구에서는 `student_strategy.py`의 `loop_decision`을 직접 작성합니다.

`starter_strategy.txt`가 VLM에게 이미지를 보고 무엇을 판단할지 지시하는 **내부 프롬프트**입니다. 공통 코드가 과제 정의·현재 이미지·JSON 응답 형식을 붙입니다. 실제 입력과 출력은 각 시험의 `plan_001.json`에서 확인합니다. 응답의 `assessments`는 판단 기록이며, 실제 실행 순서는 `plan` 배열입니다.


### A: VLA 단독


In [ ]:
if 'A' in RUN_CONDITIONS:
    trial('A')


### B: VLM+VLA 1회


In [ ]:
if 'B' in RUN_CONDITIONS:
    trial('B')


### C: VLM–VLA 피드백 루프


In [ ]:
if 'C' in RUN_CONDITIONS:
    trial('C')


### D: D-state 고전 제어


In [ ]:
if 'D' in RUN_CONDITIONS:
    trial('D')


### E: Windows 로컬 실행

Colab의 pygame은 로컬 USB 장치를 읽을 수 없습니다. 같은 씬 JSON을 로컬로 복사하고 로컬 커널에서 실행하세요. Enter/녹화 버튼으로 시작, Esc로 종료합니다. Space는 그리퍼입니다. 리셋은 무효입니다. 상단·손목 2개 카메라를 표시합니다. 연습 시간·조작자·시험 순서를 연구 노트에 남기세요.


In [ ]:
RUN_HUMAN = False
if RUN_HUMAN:
    if IS_COLAB:
        raise RuntimeError('E는 Windows 로컬 커널에서 실행하세요.')
    args = ['trial', '--condition', 'E', '--scene', SCENE,
            '--out', TRIALS/Path(SCENE).stem/'E', '--seconds', SECONDS,
            '--max-steps', MAX_STEPS, '--operator', 'operator01', '--input-kind', 'gamepad']
    if TASK_COMMAND.strip():
        args += ['--task-command', TASK_COMMAND]
    run.run('scripts/final_project_v2.py', *args)


## 결과 취합·영상

오류 시험도 남깁니다. 모델·학생 코드·장비·예산을 기록하며 장비가 다른 E는 다른 cohort로 표시합니다. `results.csv`: 개별 시험, `summary.json`: 평균·같은 씬의 차이·미실행 확인, `index.html`: 산출물 링크.

MP4는 **시뮬레이션 시간**으로 재생합니다. 추론 대기를 포함한 실제 시간은 CSV·영상의 wall 표시·events·trace에서 확인하세요. 영상 생성·백업 시간은 과제 시간에서 제외합니다.


In [ ]:
run.run('scripts/final_project_v2.py', 'aggregate', '--root', TRIALS)
display(json.loads((TRIALS/'comparison/summary.json').read_text(encoding='utf-8')))
for report_file in sorted(TRIALS.rglob('report.json')):
    report = json.loads(report_file.read_text(encoding='utf-8'))
    print(report['condition'], report['status'], report['final']['acc'], report['final']['macro_f1'])
    if (report_file.parent/'process.mp4').is_file():
        show_video(report_file.parent/'process.mp4')
run.sync()


## 여러 씬의 독립 실행

개발은 `development`, 모델·전략 선택은 `validation`, 마지막은 `test`입니다. 같은 suite로 조건별 독립 실행 후 결과를 합칠 수 있습니다. 정책 seed 반복은 독립 씬 개수가 아닙니다. 출력은 덮어쓰지 않습니다.


In [ ]:
RUN_SUITE = os.environ.get('FPV2_RUN_SUITE', '0') == '1'  # 3개 이상 씬 실행은 명시 선택
SPLIT = 'validation'
if RUN_SUITE:
    check_research_budget(RUN_CONDITIONS, SECONDS, MAX_STEPS, ACTION_STEPS, MAX_CALLS, smoke=SMOKE_TEST)
    SUITE_TRIALS = run.root/('suite_trials_'+uuid.uuid4().hex[:8])
    if not any(r['split'] == SPLIT for r in registry['scenes']):
        raise ValueError('선택한 SPLIT의 씬이 없습니다: '+SPLIT)
    args = ['suite', '--scenes', SCENES, '--split', SPLIT, '--conditions', ','.join(RUN_CONDITIONS),
            '--out', SUITE_TRIALS, '--seconds', SECONDS, '--max-steps', MAX_STEPS,
            '--action-steps', ACTION_STEPS, '--observer-steps', OBSERVER_STEPS,
            '--max-calls', MAX_CALLS, '--repeat-seeds', REPEAT_SEEDS,
            '--model-source', MODEL_SOURCE, '--models-dir', MODELS_DIR,
            '--vlm-source', VLM_SOURCE, '--vla-source', VLA_SOURCE]
    if VLA_SUBTASK:
        args += ['--checkpoint', VLA_SUBTASK]
    if VLA_SCENE:
        args += ['--a-checkpoint', VLA_SCENE]
    if VLM_MODEL and any(c in 'BC' for c in RUN_CONDITIONS):
        args += ['--vlm-model', VLM_MODEL]
    if 'C' in RUN_CONDITIONS:
        args += ['--strategy', C_STRATEGY]
    if 'B' in RUN_CONDITIONS:
        args += ['--b-strategy', B_STRATEGY]
    if TASK_COMMAND.strip():
        args += ['--task-command', TASK_COMMAND]
    if FINAL_TEST:
        args += ['--final-test']
    run.run('scripts/final_project_v2.py', *args)

    display(json.loads((SUITE_TRIALS/'comparison/summary.json').read_text(encoding='utf-8')))
